# 📓 Notebook 10 — Planning Agents---**Phase 4 · Agent Architecture**> A ReAct agent reacts step-by-step. A planning agent thinks AHEAD — decomposes the task, then executes.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| Task decomposition | Breaking complex tasks into sub-tasks || Dependency graphs | Understanding which tasks depend on others || Planner-Executor | Separate planning from execution || Re-planning | Adapting the plan when things go wrong |### 🏗️ Build: Agent That Plans, Executes, and Reflects

## 1. Setup

In [ ]:
import os, sys
import litellm

from setup_llm import DEFAULT_MODEL, is_proxy_mode

mode = "🔗 proxy" if is_proxy_mode() else "🔑 direct"
print(f"🤖 Model: {DEFAULT_MODEL} ({mode})")

---## 2. Task Decomposition### The Core IdeaInstead of solving a problem step-by-step reactively, **plan all steps upfront** then execute.```Complex Task    ↓┌── Planner ──────────────────┐│  1. Extract data from PDF   ││  2. Analyze key metrics     ││  3. Compare with benchmarks ││  4. Generate report         │└──────────────────────────────┘    ↓┌── Executor ─────────────────┐│  Step 1: [execute] → result ││  Step 2: [execute] → result ││  Step 3: [execute] → result ││  Step 4: [execute] → result │└──────────────────────────────┘```### Planner-Executor vs ReAct| Aspect | Planner-Executor | ReAct ||--------|-----------------|-------|| Planning | Upfront, complete | None, reactive || Adaptability | Re-plan needed | Naturally adaptive || Efficiency | Can parallelize steps | Sequential only || Best for | Well-structured tasks | Exploratory tasks |

### 🖼️ Task Dependencies as a DAG

![Task Decomposition as a DAG](images/planning_agent_dag.png)

#### 💡 Real-World Analogy: Cooking a Meal

Task decomposition works like **planning a dinner party**:

```
Goal: "Prepare pasta dinner for 4 guests"

Step 1: Boil water         (no dependencies → start immediately) 
Step 2: Chop vegetables    (no dependencies → start immediately)
Step 3: Cook pasta         (depends on Step 1 — water must boil first)
Step 4: Make sauce         (depends on Step 2 — veggies must be chopped)
Step 5: Combine & plate    (depends on Steps 3 AND 4)
Step 6: Serve              (depends on Step 5)
```

Notice: Steps 1 & 2 can happen **at the same time** (parallel execution!).
But Step 5 must wait for BOTH pasta AND sauce to be ready.

A planning agent uses exactly this logic:
- **No dependencies?** → Run now (or in parallel)
- **Has dependencies?** → Wait until they complete
- **Step fails?** → Re-plan around the failure

> This is the same algorithm used in build tools (Make, Gradle) and CI/CD pipelines — **topological sorting** of a DAG.

In [ ]:
from pydantic import BaseModel, Fieldfrom typing import List, Optionalclass TaskStep(BaseModel):    step_id: int    description: str    depends_on: List[int] = Field(default_factory=list)    tool_needed: Optional[str] = None    status: str = "pending"  # pending, running, completed, failed    result: Optional[str] = Noneclass TaskPlan(BaseModel):    goal: str    steps: List[TaskStep]        def get_ready_steps(self):        """Get steps whose dependencies are all completed."""        completed = {s.step_id for s in self.steps if s.status == "completed"}        return [s for s in self.steps if s.status == "pending" and all(d in completed for d in s.depends_on)]        def is_complete(self):        return all(s.status == "completed" for s in self.steps)        def display(self):        print(f"📋 Plan: {self.goal}")        for s in self.steps:            icon = {"pending": "⬜", "running": "🔄", "completed": "✅", "failed": "❌"}[s.status]            deps = f" (after: {s.depends_on})" if s.depends_on else ""            print(f"   {icon} Step {s.step_id}: {s.description}{deps}")print("✅ Task models defined")

In [ ]:
class Planner:    """Decomposes complex tasks into executable steps."""        def __init__(self, model=None):        self.model = model or DEFAULT_MODEL        def plan(self, goal):        """Generate a task plan from a goal."""        response = litellm.completion(            model=self.model,            messages=[{                "role": "system",                "content": """You are a task planner. Decompose the given goal into 3-7 concrete, actionable steps.For each step specify:- step_id (integer starting from 1)- description (clear, actionable)- depends_on (list of step_ids that must complete first, empty if none)- tool_needed (one of: "search", "calculate", "analyze", "write", null)Return as JSON with key "steps" containing the array."""            }, {                "role": "user",                "content": f"Goal: {goal}"            }],            response_format={"type": "json_object"},            temperature=0,        )                data = json.loads(response.choices[0].message.content)        steps = [TaskStep(**s) for s in data["steps"]]        return TaskPlan(goal=goal, steps=steps)# Demoplanner = Planner()plan = planner.plan("Research and compare Python vs Rust for building a high-frequency trading system")plan.display()

In [ ]:
class Executor:    """Executes individual plan steps."""        def __init__(self, model=None):        self.model = model or DEFAULT_MODEL        def execute_step(self, step, context=""):        """Execute a single step given accumulated context."""        response = litellm.completion(            model=self.model,            messages=[{                "role": "system",                "content": "You are an executor that carries out specific tasks. Be thorough and provide detailed, useful output."            }, {                "role": "user",                "content": f"Task: {step.description}\n\nContext from previous steps:\n{context if context else 'None'}\n\nExecute this task and provide the result."            }],            temperature=0.3, max_tokens=500,        )        return response.choices[0].message.contentclass PlannerExecutor:    """Complete Planner-Executor agent."""        def __init__(self, model=None):        self.planner = Planner(model)        self.executor = Executor(model)        def run(self, goal, verbose=True):        """Plan and execute a complex task."""        # Phase 1: Plan        if verbose:            print("🧠 PHASE 1: Planning")            print("=" * 50)                plan = self.planner.plan(goal)        if verbose:            plan.display()                # Phase 2: Execute        if verbose:            print(f"\n⚡ PHASE 2: Execution")            print("=" * 50)                context_parts = []                while not plan.is_complete():            ready = plan.get_ready_steps()            if not ready:                print("⚠️ No ready steps — possible circular dependency!")                break                        for step in ready:                step.status = "running"                if verbose:                    print(f"\n🔄 Executing Step {step.step_id}: {step.description}")                                try:                    context = "\n".join(context_parts[-3:])  # Last 3 results                    result = self.executor.execute_step(step, context)                    step.result = result                    step.status = "completed"                    context_parts.append(f"Step {step.step_id} ({step.description}): {result[:200]}")                                        if verbose:                        print(f"   ✅ Done: {result[:100]}...")                except Exception as e:                    step.status = "failed"                    step.result = str(e)                    if verbose:                        print(f"   ❌ Failed: {e}")                # Phase 3: Synthesize        if verbose:            print(f"\n📝 PHASE 3: Synthesis")            print("=" * 50)                all_results = "\n\n".join([            f"Step {s.step_id} ({s.description}):\n{s.result}"             for s in plan.steps if s.status == "completed"        ])                synthesis = litellm.completion(            model=self.planner.model,            messages=[{                "role": "user",                "content": f"Goal: {goal}\n\nResults from all steps:\n{all_results}\n\nSynthesize a comprehensive final answer."            }],            temperature=0.3, max_tokens=800,        )                final = synthesis.choices[0].message.content        if verbose:            print(f"\n{final}")                return {"plan": plan, "final_answer": final}# Run the planner-executorpe = PlannerExecutor()result = pe.run("Research and compare Python vs Rust for building a high-frequency trading system")

---## 3. Re-Planning (Adaptive Planning)When a step fails or produces unexpected results, the agent re-plans.

In [ ]:
class AdaptivePlannerExecutor(PlannerExecutor):    """Planner-Executor that can re-plan on failure."""        def run(self, goal, max_replans=2, verbose=True):        for attempt in range(max_replans + 1):            if verbose and attempt > 0:                print(f"\n🔄 RE-PLANNING (attempt {attempt + 1})")                        result = super().run(goal, verbose=verbose)                        # Check if plan completed successfully            failed = [s for s in result["plan"].steps if s.status == "failed"]            if not failed:                return result                        # Re-plan with failure context            failure_info = "\n".join([f"Step {s.step_id} failed: {s.result}" for s in failed])            goal = f"{goal}\n\nPrevious attempt had these failures:\n{failure_info}\nPlease create an alternative plan."                return resultadaptive = AdaptivePlannerExecutor()# Would re-plan if any step failsprint("✅ Adaptive Planner-Executor ready")

---## 📝 Interview Quiz — Planning Agents### Q1: Planner-Executor vs ReAct — when would you use each?<details><summary>💡 Answer</summary>**Planner-Executor when:**- Task is well-defined (report generation, data pipeline)- Steps can be determined upfront- Parallelization is beneficial- You want a clear audit trail**ReAct when:**- Task is exploratory (research, debugging)- Next step depends on current results- Problem structure is unknown upfront- Flexibility is more important than efficiency</details>### Q2: How do you handle task dependencies in a planning agent?<details><summary>💡 Answer</summary>Model as a **DAG (Directed Acyclic Graph)**:- Each step has `depends_on` — list of prerequisite step IDs- A step is "ready" when all its dependencies are completed- Steps with no dependencies can run first (or in parallel)- Detect cycles to prevent deadlocks- Failed dependencies should cascade-fail dependent stepsThis is essentially topological sorting — the same algorithm used in build systems (Make, Gradle).</details>### Q3: What happens when a plan step fails? How do you recover?<details><summary>💡 Answer</summary>Recovery strategies (from simplest to most complex):1. **Retry** — Same step, same approach (transient failures)2. **Retry with modification** — Send error to LLM, ask for alternative approach3. **Skip and continue** — If step is non-critical4. **Re-plan from failure point** — Re-generate remaining steps given partial results5. **Full re-plan** — Generate entirely new plan with failure context6. **Escalate** — Ask human for help**Best practice:** Start with retry (2-3 attempts), then re-plan, then escalate.</details>### Q4: How would you implement parallel step execution?<details><summary>💡 Answer</summary>```pythonimport asyncioready_steps = plan.get_ready_steps()  # Steps with all deps met# These can run in parallel since they don't depend on each othertasks = [execute_step_async(step) for step in ready_steps]results = await asyncio.gather(*tasks)```**Key considerations:**- Only parallelize steps where all dependencies are met- Rate limit API calls to avoid throttling- Handle partial failures (some parallel steps succeed, others fail)- Update dependency graph after each parallel batch completes</details>

---## ✅ Notebook 10 Summary| Concept | Key Takeaway ||---------|-------------|| Task decomposition | LLM breaks complex goal into steps with dependencies || Dependency graph | DAG of steps; topological ordering for execution || Planner-Executor | Separate planning from execution for clarity || Re-planning | Detect failures and generate alternative plans || vs ReAct | Planning = structured efficiency; ReAct = flexible exploration |### ➡️ Next: [Notebook 11 — Multi-Agent Systems](./11_multi_agent.ipynb)